# 05 Evaluate AE Scores and Thresholds

Compute reconstruction scores for the trained convolutional autoencoder, select thresholds only on validation data, and evaluate the held-out test split with frozen thresholds.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, precision_recall_curve
import tensorflow as tf


In [ ]:
REPO_ROOT = Path('..').resolve()
MANIFEST_PATH = REPO_ROOT / 'reports' / 'manifests' / 'turning_split_seed42.csv'
MODEL_PATH = REPO_ROOT / 'models' / 'ae_bn16_20260616.keras'
THRESHOLD_DIR = REPO_ROOT / 'reports' / 'thresholds'
SCORE_DIR = REPO_ROOT / 'reports' / 'scores'
TABLE_DIR = REPO_ROOT / 'reports' / 'tables'

IMAGE_SIZE = (150, 100)
N_VER_SEGMENTS = 10
VER_TOP_K = 3

THRESHOLD_DIR.mkdir(parents=True, exist_ok=True)
SCORE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
manifest = manifest[(manifest['source_dataset'] == 'turning') & manifest['split'].isin(['validation', 'test'])].copy()
manifest = manifest[manifest['image_path'].fillna('').ne('')].copy()
manifest['target'] = (manifest['label'] == 'chatter').astype(int)
manifest.groupby(['split', 'label']).size().rename('n').reset_index()


In [ ]:
def load_images(rows: pd.DataFrame, image_size=IMAGE_SIZE) -> tuple[np.ndarray, pd.DataFrame]:
    images = []
    loaded_rows = []
    for _, row in rows.iterrows():
        image_path = REPO_ROOT / row['image_path']
        if not image_path.exists():
            print(f'Missing image, skipping: {image_path}')
            continue
        image = Image.open(image_path).convert('RGB').resize(image_size)
        images.append(np.asarray(image, dtype='float32') / 255.0)
        loaded_rows.append(row)
    if not images:
        raise ValueError('No images loaded. Run notebooks 02 and 03 first.')
    return np.stack(images, axis=0), pd.DataFrame(loaded_rows).reset_index(drop=True)

validation_images, validation_rows = load_images(manifest[manifest['split'] == 'validation'])
test_images, test_rows = load_images(manifest[manifest['split'] == 'test'])
print('validation:', validation_images.shape)
print('test:', test_images.shape)


In [ ]:
def vertical_segment_scores(error_map: np.ndarray, n_segments: int, top_k: int) -> tuple[np.ndarray, np.ndarray]:
    width = error_map.shape[2]
    boundaries = np.linspace(0, width, n_segments + 1, dtype=int)
    segment_scores = []
    for left, right in zip(boundaries[:-1], boundaries[1:]):
        if right <= left:
            continue
        segment_scores.append(error_map[:, :, left:right, :].mean(axis=(1, 2, 3)))
    segment_scores = np.stack(segment_scores, axis=1)
    ver_max = segment_scores.max(axis=1)
    top_k = min(top_k, segment_scores.shape[1])
    ver_topk = np.sort(segment_scores, axis=1)[:, -top_k:].mean(axis=1)
    return ver_max, ver_topk

def score_reconstructions(model, images: np.ndarray) -> pd.DataFrame:
    recon = model.predict(images, verbose=0)
    squared_error = (images - recon) ** 2
    absolute_error = np.abs(images - recon)
    ver_max, ver_topk = vertical_segment_scores(squared_error, N_VER_SEGMENTS, VER_TOP_K)
    return pd.DataFrame({
        'global_mse': squared_error.mean(axis=(1, 2, 3)),
        'global_mae': absolute_error.mean(axis=(1, 2, 3)),
        'ver_max': ver_max,
        'ver_topk': ver_topk,
    })


In [ ]:
model = tf.keras.models.load_model(MODEL_PATH)
validation_scores = pd.concat([validation_rows.reset_index(drop=True), score_reconstructions(model, validation_images)], axis=1)
test_scores = pd.concat([test_rows.reset_index(drop=True), score_reconstructions(model, test_images)], axis=1)
all_scores = pd.concat([validation_scores, test_scores], ignore_index=True)
SCORES_PATH = SCORE_DIR / 'ae_scores_turning.csv'
all_scores.to_csv(SCORES_PATH, index=False)
print(f'Wrote {SCORES_PATH}')


In [ ]:
def select_best_f1_threshold(y_true: np.ndarray, scores: np.ndarray) -> dict:
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    if len(thresholds) == 0:
        threshold = float(np.max(scores))
        return {'threshold': threshold, 'validation_f1': 0.0, 'validation_precision': 0.0, 'validation_recall': 0.0}
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    best_idx = int(np.nanargmax(f1))
    return {
        'threshold': float(thresholds[best_idx]),
        'validation_f1': float(f1[best_idx]),
        'validation_precision': float(precision[best_idx]),
        'validation_recall': float(recall[best_idx]),
    }

def evaluate_at_threshold(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'pr_auc': float(average_precision_score(y_true, scores)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }


In [ ]:
score_columns = ['global_mse', 'global_mae', 'ver_max', 'ver_topk']
y_val = validation_scores['target'].to_numpy()
y_test = test_scores['target'].to_numpy()
thresholds = {}
metric_rows = []

for score_name in score_columns:
    selected = select_best_f1_threshold(y_val, validation_scores[score_name].to_numpy())
    thresholds[score_name] = selected
    test_metrics = evaluate_at_threshold(y_test, test_scores[score_name].to_numpy(), selected['threshold'])
    metric_rows.append({
        'dataset': 'turning',
        'method': 'cnn_ae',
        'score': score_name,
        'threshold': selected['threshold'],
        **test_metrics,
    })

THRESHOLD_PATH = THRESHOLD_DIR / 'ae_thresholds_turning.json'
METRICS_PATH = TABLE_DIR / 'metrics_turning_ae.csv'
with THRESHOLD_PATH.open('w') as f:
    json.dump(thresholds, f, indent=2)
pd.DataFrame(metric_rows).to_csv(METRICS_PATH, index=False)
print(f'Wrote {THRESHOLD_PATH}')
print(f'Wrote {METRICS_PATH}')
pd.DataFrame(metric_rows)
